# pic drawings on a map

The four `examples/geo` sources, georeferenced with `rpic.geo` and shown on a Leaflet map through **folium**. Needs `pip install "rpiclang[geo]" folium`.

In [ ]:
import pathlib

import folium
import rpic

here = pathlib.Path('.')
sources = ['callout', 'arrow-between-features', 'radius', 'two-control-points']
print(rpic.__version__)

In [ ]:
def style(feature):
    p = feature['properties']
    return {
        'color': p.get('stroke', '#000000'),
        'weight': p.get('stroke-width', 0.8) * 1.5,
        'fillColor': p.get('fill', '#ffffff'),
        'fillOpacity': 0.6 if 'fill' in p else 0.0,
        'dashArray': '6 4' if 'geopic:dash' in p else None,
    }

m = folium.Map(location=[-8.045, -34.87], zoom_start=13, tiles='CartoDB positron')
for name in sources:
    out = rpic.geo.compile((here / f'{name}.pic').read_text(), warn=False)
    for w in out['warnings']:
        print(name, w['kind'], w['message'])
    layer = folium.GeoJson(
        out['geojson'],
        name=name,
        style_function=style,
        tooltip=folium.GeoJsonTooltip(fields=['geopic:kind', 'title'], aliases=['kind', 'text'], labels=True),
    )
    layer.add_to(m)
    r = out['registration']
    print(f"{name:26s} {r['method']:10s} {r['scale_m_per_in']:8.1f} m/in  bearing {r['bearing_deg']:7.2f}°")
folium.LayerControl().add_to(m)
m

Labels (`geopic:kind == "text"`) are points; a real app would render them as map labels using `geopic:text`, `geopic:anchor` and `geopic:rotation` (degrees counter-clockwise from east).

In [ ]:
gdf = rpic.geo.to_geodataframe((here / 'two-control-points.pic').read_text(), warn=False)
gdf[['geopic:id', 'geopic:kind', 'title', 'fill', 'geometry']]